# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked Action Queue

The action queue ranks pages using the model score and assigns a human-readable reason code.

The purpose is to help a content team decide which pages to inspect first.

The action labels are:

- `review_refresh` — inspect the page for freshness, relevance, and content-quality opportunities.
- `review_visibility` — inspect search visibility and query/page alignment.
- `monitor` — retain the page for monitoring rather than prioritizing immediate review.

The ranking is a decision-support output. A high score does not automatically mean that a page should be changed.

In [3]:
# ============================================================
# W07 — Setup + Build Ranked Action Queue
# ============================================================

import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 1. Connect to FlyRank warehouse
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

# Hide DuckDB progress bars
con.execute("SET enable_progress_bar = false")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret
(
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret
(
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connection: READY")
print("FlyRank warehouse: READY")


# ------------------------------------------------------------
# 2. Build February feature table
# ------------------------------------------------------------

feb_features = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{FEB}')
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days
FROM feb f
JOIN read_parquet('{DIM_CONTENT}') c
    ON f.client_hash_id = c.client_hash_id
   AND f.content_hash_id = c.content_hash_id
WHERE c.content_created_date IS NOT NULL
""").df()

print("February feature rows:", len(feb_features))


# ------------------------------------------------------------
# 3. Build March observed outcome
# ------------------------------------------------------------

march_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 1
        ELSE 0
    END AS went_dark
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("March outcome rows:", len(march_outcome))


# ------------------------------------------------------------
# 4. Join February features with March outcome
# ------------------------------------------------------------

model_df = feb_features.merge(
    march_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df = model_df.dropna(
    subset=[
        "gsc_impressions",
        "gsc_clicks",
        "content_age_days",
        "went_dark"
    ]
).copy()

print("Final modeling rows:", len(model_df))


# ------------------------------------------------------------
# 5. Define model features and target
# ------------------------------------------------------------

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

X = model_df[feature_cols]
y = model_df["went_dark"]
groups = model_df["client_hash_id"]


# ------------------------------------------------------------
# 6. Honest grouped train/test split
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


# ------------------------------------------------------------
# 7. Train Logistic Regression
# ------------------------------------------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic_regression",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

print("Logistic Regression: TRAINED")


# ------------------------------------------------------------
# 8. Score the held-out test population
# ------------------------------------------------------------

test_results = model_df.iloc[test_idx].copy()

test_results["model_score"] = model.predict_proba(
    X_test
)[:, 1]


# ------------------------------------------------------------
# 9. Recreate transparent Week-4 baseline score
# ------------------------------------------------------------

def baseline_score(row):

    age = row["content_age_days"]
    impressions = row["gsc_impressions"]

    score = 0

    if age >= 365:
        score += 2
    elif age >= 180:
        score += 1

    if impressions < 100:
        score += 2
    elif impressions < 1000:
        score += 1

    return score


test_results["baseline_score"] = test_results.apply(
    baseline_score,
    axis=1
)


# ------------------------------------------------------------
# 10. Create reason codes
# ------------------------------------------------------------

def make_reason(row):

    age = row["content_age_days"]
    impressions = row["gsc_impressions"]

    if age >= 365 and impressions < 100:
        return "stale_low_visibility"

    if age >= 180 and impressions < 1000:
        return "aging_low_visibility"

    if impressions < 100:
        return "low_visibility"

    if age >= 180:
        return "aging_content"

    return "visible_low_priority"


test_results["reason_code"] = test_results.apply(
    make_reason,
    axis=1
)


# ------------------------------------------------------------
# 11. Create human-review actions
# ------------------------------------------------------------

def make_action(row):

    reason = row["reason_code"]

    if reason in [
        "stale_low_visibility",
        "aging_low_visibility"
    ]:
        return "review_refresh"

    if reason == "low_visibility":
        return "review_visibility"

    if reason == "aging_content":
        return "review_refresh"

    return "monitor"


test_results["action"] = test_results.apply(
    make_action,
    axis=1
)


# ------------------------------------------------------------
# 12. Rank the queue
# ------------------------------------------------------------

action_queue = (
    test_results[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "baseline_score",
            "content_age_days",
            "gsc_impressions",
            "gsc_clicks",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        by=[
            "model_score",
            "baseline_score"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

action_queue.insert(
    0,
    "rank",
    range(1, len(action_queue) + 1)
)


# ------------------------------------------------------------
# 13. Display top 20
# ------------------------------------------------------------

display(
    action_queue.head(20)
)

print("Ranked action queue: READY")
print("Queue rows:", len(action_queue))

DuckDB connection: READY
FlyRank warehouse: READY
February feature rows: 153559
March outcome rows: 176738
Final modeling rows: 134238
Training rows: 88344
Test rows: 45894
Logistic Regression: TRAINED


,rank,client_hash_id,content_hash_id,model_score,baseline_score,content_age_days,gsc_impressions,gsc_clicks,reason_code,action
0,1,client_9958f0a7ae1df715,content_29caac8293c2b5fe,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
1,2,client_9958f0a7ae1df715,content_6322a23d23f77a1f,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
2,3,client_9958f0a7ae1df715,content_f14be505493f6c2b,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
3,4,client_9958f0a7ae1df715,content_ad96b6464e66019f,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
4,5,client_9958f0a7ae1df715,content_3161f3d25837193d,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
5,6,client_9958f0a7ae1df715,content_42678c4ab22f4294,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
6,7,client_9958f0a7ae1df715,content_9184c40fede11487,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
7,8,client_9958f0a7ae1df715,content_30e68e15ee48f40a,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
8,9,client_9958f0a7ae1df715,content_209380decb3dee8a,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
9,10,client_9958f0a7ae1df715,content_326252045e43eda4,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh


Ranked action queue: READY
Queue rows: 45894


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

The action playbook is intended for a content or SEO team that needs to prioritize limited review capacity.

A reviewer can use the ranking to decide which pages to inspect first and then combine the model signal with page context, search intent, business importance, seasonality, and editorial judgment.

### Limits

The model score is not an instruction to automatically refresh, remove, or rewrite a page.

The observed `went_dark` outcome is a proxy based on March GSC clicks and does not measure whether a refresh would succeed.

The ranking is based on the available features and evaluation population, so its performance may differ for other clients, time periods, or future data.

In [8]:
action_summary = (
    action_queue
    .groupby("action")
    .agg(
        pages=("rank", "count"),
        average_model_score=("model_score", "mean"),
        average_content_age_days=("content_age_days", "mean"),
        average_impressions=("gsc_impressions", "mean")
    )
    .reset_index()
    .sort_values(
        "average_model_score",
        ascending=False
    )
)

display(action_summary)

print("Action summary: READY")

,action,pages,average_model_score,average_content_age_days,average_impressions
2,review_visibility,9680,0.754341,52.689876,25.786674
1,review_refresh,28498,0.590131,289.145344,1349.587339
0,monitor,7716,0.367832,81.095257,2359.014386


Action summary: READY


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Requirements

Before taking any content action, a human reviewer should check:

- Whether the page still matches the intended search intent.
- Whether the topic is still relevant to the audience.
- Whether the content is accurate and up to date.
- Whether the page has seasonal or temporary demand patterns.
- Whether the page has important business or editorial context.
- Whether the observed search signals are sufficient to justify review.
- Whether the proposed action is appropriate after inspecting the actual page.

### No-Go List

The following decisions should not be automated from this ranking alone:

- Automatically deleting a page.
- Automatically rewriting or publishing content.
- Automatically changing search strategy.
- Automatically declaring a page unsuccessful.
- Automatically treating a high score as proof that a refresh will improve performance.
- Automatically making business or editorial decisions without human review.

In [9]:
# ============================================================
# W07 — Human Review Gate
# ============================================================

review_queue = action_queue.head(20).copy()

review_queue["human_review_required"] = True

review_queue["review_checks"] = (
    "Check business relevance, search intent, seasonality, "
    "content quality, and whether low visibility is intentional."
)

review_queue["no_go_without_review"] = (
    "Do not automatically delete, rewrite, publish, "
    "or prune content based only on the model score."
)

display(
    review_queue[
        [
            "rank",
            "model_score",
            "baseline_score",
            "content_age_days",
            "gsc_impressions",
            "gsc_clicks",
            "reason_code",
            "action",
            "human_review_required",
            "review_checks",
            "no_go_without_review"
        ]
    ]
)

print("Human review gate: READY")

,rank,model_score,baseline_score,content_age_days,gsc_impressions,gsc_clicks,reason_code,action,human_review_required,review_checks,no_go_without_review
0,1,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
1,2,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
2,3,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
3,4,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
4,5,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
5,6,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
6,7,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
7,8,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
8,9,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."
9,10,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh,True,"Check business relevance, search intent, seaso...","Do not automatically delete, rewrite, publish,..."


Human review gate: READY


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and Retraining Triggers

The recommendations should be reviewed if the underlying data or measured model performance changes.

Potential triggers include:

- A sustained change in the distribution of impressions, clicks, or content age.
- A measurable decline in Precision@50 or Average Precision on a later evaluation window.
- A change in the relationship between the model ranking and the observed outcome.
- A change in the available data schema or feature definitions.
- A new period or client population where the current validation results no longer represent the intended use.

A later evaluation should use a new time window and should keep future outcomes separate from decision-time features.

The model should be retrained or revalidated when monitoring shows that its measured ranking quality is no longer adequate for the intended decision-support use.

In [10]:

# ============================================================
# W07 — Monitoring and Retrain Checks
# ============================================================

monitoring = pd.DataFrame({
    "metric": [
        "Test population size",
        "Observed went_dark rate",
        "Top-50 average model score",
        "Top-50 baseline average score",
        "Minimum model score in top 20",
        "Maximum model score in top 20"
    ],
    "value": [
        len(test_results),
        test_results["went_dark"].mean(),
        action_queue.head(50)["model_score"].mean(),
        action_queue.head(50)["baseline_score"].mean(),
        action_queue.head(20)["model_score"].min(),
        action_queue.head(20)["model_score"].max()
    ]
})

display(monitoring)

print("Monitoring checks: READY")
print()
print("Suggested retrain triggers:")
print("- Material drop in Precision@50")
print("- Material drop in Average Precision")
print("- Major change in feature distributions")
print("- Major change in data availability")
print("- Change in meaning or definition of input fields")

,metric,value
0,Test population size,45894.000000
1,Observed went_dark rate,0.622565
2,Top-50 average model score,0.823638
3,Top-50 baseline average score,4.000000
4,Minimum model score in top 20,0.823752
5,Maximum model score in top 20,0.823752


Monitoring checks: READY

Suggested retrain triggers:
- Material drop in Precision@50
- Material drop in Average Precision
- Major change in feature distributions
- Major change in data availability
- Change in meaning or definition of input fields


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Public-Safe Paper Exports

The action queue is exported to `work/outputs/` so that the research paper can reuse the analysis output.

The public-facing recommendation table uses recommendation IDs rather than client or content identifiers.

The exported artifacts contain model and signal information needed to explain the ranking while avoiding private client information.

In [11]:
# ============================================================
# W07 — Export Public-Safe Queue
# ============================================================

import os

os.makedirs("work/outputs", exist_ok=True)

# ------------------------------------------------------------
# 1. Full internal queue
# ------------------------------------------------------------

internal_export_path = "work/outputs/action_playbook_queue.csv"

action_queue.to_csv(
    internal_export_path,
    index=False
)

print("Internal queue exported:", internal_export_path)


# ------------------------------------------------------------
# 2. Public-safe top 20
# ------------------------------------------------------------

public_queue = action_queue.head(20).copy()

public_queue = public_queue[
    [
        "rank",
        "model_score",
        "baseline_score",
        "content_age_days",
        "gsc_impressions",
        "gsc_clicks",
        "reason_code",
        "action"
    ]
].copy()

public_export_path = "work/outputs/public_safe_action_queue.csv"

public_queue.to_csv(
    public_export_path,
    index=False
)

print("Public-safe export:", public_export_path)
print("Rows exported:", len(public_queue))


# ------------------------------------------------------------
# 3. Verify public-safe output
# ------------------------------------------------------------

forbidden_columns = [
    "client_hash_id",
    "content_hash_id"
]

assert not any(
    column in public_queue.columns
    for column in forbidden_columns
)

assert len(public_queue) == 20

assert public_queue["rank"].is_unique

assert public_queue["rank"].min() == 1

print("Public-safe export check: PASSED")
print("Paper export: READY")

display(public_queue)

Internal queue exported: work/outputs/action_playbook_queue.csv
Public-safe export: work/outputs/public_safe_action_queue.csv
Rows exported: 20
Public-safe export check: PASSED
Paper export: READY


,rank,model_score,baseline_score,content_age_days,gsc_impressions,gsc_clicks,reason_code,action
0,1,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
1,2,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
2,3,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
3,4,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
4,5,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
5,6,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
6,7,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
7,8,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
8,9,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
9,10,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.